### 課題１

In [15]:
# import sample data: Loan screening data for classification 
import pandas as pd

df = pd.read_csv('./data/final_hr_analysis_train.csv', header=0)

# check the shape
print('Raw shape: (%i,%i)' %df.shape)

df.head()

Raw shape: (10499,11)


,index,left,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales,salary
0,10438,0,0.53,0.52,2,135,4,0,0,technical,medium
1,9236,0,0.77,0.53,5,256,3,0,0,accounting,medium
2,818,1,0.89,0.79,3,149,2,0,0,support,medium
3,11503,0,0.64,0.63,3,156,6,1,0,support,low
4,11721,0,0.98,0.74,4,151,3,0,0,sales,medium


In [16]:
X  = df.iloc[:,2:]           # 3列目以降を特徴量X
ID = df.iloc[:,[0]]          # 1列目をID情報としてセット
y  = df.iloc[:,1]            # 2列目を正解データ

# check the shape
print('X shape: (%i,%i)' %X.shape)
print('---------------------------------------')
print('y values:')
print(y.value_counts())
print('---------------------------------------')

# データ型の確認
print(ID.join(X).join(y).dtypes)
ID.join(X).join(y).head()

X shape: (10499,9)
---------------------------------------
y values:
0    7966
1    2533
Name: left, dtype: int64
---------------------------------------
index                      int64
satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
promotion_last_5years      int64
sales                     object
salary                    object
left                       int64
dtype: object


,index,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales,salary,left
0,10438,0.53,0.52,2,135,4,0,0,technical,medium,0
1,9236,0.77,0.53,5,256,3,0,0,accounting,medium,0
2,818,0.89,0.79,3,149,2,0,0,support,medium,1
3,11503,0.64,0.63,3,156,6,1,0,support,low,0
4,11721,0.98,0.74,4,151,3,0,0,sales,medium,0


In [17]:
#one-hotエンコーディング
ohe_columns = ['sales',
               'salary']
X_ohe = pd.get_dummies(X,
                       dummy_na=True,
                       columns=ohe_columns)
print('X_ohe shape:(%i,%i)' % X_ohe.shape)
X_ohe.head()

X_ohe shape:(10499,22)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales_IT,sales_RandD,sales_accounting,...,sales_marketing,sales_product_mng,sales_sales,sales_support,sales_technical,sales_nan,salary_high,salary_low,salary_medium,salary_nan
0,0.53,0.52,2,135,4,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
1,0.77,0.53,5,256,3,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
2,0.89,0.79,3,149,2,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
3,0.64,0.63,3,156,6,1,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,0.98,0.74,4,151,3,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0


In [18]:
from sklearn.preprocessing import Imputer

# 欠損値NaNを平均値(mean)で置換
imp = Imputer(missing_values='NaN', strategy='mean', axis=0)
imp.fit(X_ohe)

# 学習済みImputerを適用しX_newの欠損値を置換
X_ohe_columns = X_ohe.columns.values
X_ohe = pd.DataFrame(imp.transform(X_ohe), columns=X_ohe_columns)

# 結果表示
X_ohe.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales_IT,sales_RandD,sales_accounting,...,sales_marketing,sales_product_mng,sales_sales,sales_support,sales_technical,sales_nan,salary_high,salary_low,salary_medium,salary_nan
0,0.53,0.52,2.0,135.0,4.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,0.77,0.53,5.0,256.0,3.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,0.89,0.79,3.0,149.0,2.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,0.64,0.63,3.0,156.0,6.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.98,0.74,4.0,151.0,3.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [19]:
# RFEによる特徴量選択を実施
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestClassifier

selector = RFE(RandomForestClassifier(random_state=1),
               n_features_to_select=10,
               step=0.05)

selector.fit(X_ohe,y)

X_fin = pd.DataFrame(selector.transform(X_ohe),
                     columns=X_ohe_columns[selector.support_])

print('X_fin shape:(%i,%i)' % X_fin.shape)
X_fin.head()

X_fin shape:(10499,10)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,sales_sales,sales_support,salary_low,salary_medium
0,0.53,0.52,2.0,135.0,4.0,0.0,0.0,0.0,0.0,1.0
1,0.77,0.53,5.0,256.0,3.0,0.0,0.0,0.0,0.0,1.0
2,0.89,0.79,3.0,149.0,2.0,0.0,0.0,1.0,0.0,1.0
3,0.64,0.63,3.0,156.0,6.0,1.0,0.0,1.0,1.0,0.0
4,0.98,0.74,4.0,151.0,3.0,0.0,1.0,0.0,0.0,1.0


#### テストデータの読み込み

In [20]:
# import sample data for classificatio
df_s = pd.read_csv('./data/final_hr_analysis_test.csv', header=0)

# check the shape
print('Raw shape: (%i,%i)' %df.shape)
df_s.head()

Raw shape: (10499,11)


,index,left,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales,salary
0,1670,NaN,0.44,0.57,2,141,3,0,0,product_mng,medium
1,13378,NaN,0.55,0.96,3,194,3,0,0,product_mng,medium
2,10233,NaN,0.72,0.67,5,210,2,0,0,management,medium
3,4719,NaN,0.96,0.75,4,177,2,0,0,IT,low
4,7003,NaN,0.96,0.54,3,198,3,0,0,support,low


In [21]:
X_s  = df.iloc[:,2:]           # 3列目以降を特徴量X
ID_s = df.iloc[:,[0]]          # 1列目をID情報としてセット

# check the shape
print('Raw shape: (%i,%i)' %df_s.shape)
print('X shape: (%i,%i)' %X_s.shape)
print('---------------------------------------')

# データ型の確認
print(X_s.dtypes)
X_s.head()

Raw shape: (4500,11)
X shape: (10499,9)
---------------------------------------
satisfaction_level       float64
last_evaluation          float64
number_project             int64
average_montly_hours       int64
time_spend_company         int64
Work_accident              int64
promotion_last_5years      int64
sales                     object
salary                    object
dtype: object


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales,salary
0,0.53,0.52,2,135,4,0,0,technical,medium
1,0.77,0.53,5,256,3,0,0,accounting,medium
2,0.89,0.79,3,149,2,0,0,support,medium
3,0.64,0.63,3,156,6,1,0,support,low
4,0.98,0.74,4,151,3,0,0,sales,medium


In [22]:
# モデリング段階と同様、one-hotエンコーディングを実施
X_ohe_s = pd.get_dummies(X_s,
                         dummy_na=True,
                         columns=ohe_columns)
print('X_ohe_s shape:(%i,%i)' % X_ohe_s.shape)
X_ohe_s.head()

X_ohe_s shape:(10499,22)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,promotion_last_5years,sales_IT,sales_RandD,sales_accounting,...,sales_marketing,sales_product_mng,sales_sales,sales_support,sales_technical,sales_nan,salary_high,salary_low,salary_medium,salary_nan
0,0.53,0.52,2,135,4,0,0,0,0,0,...,0,0,0,0,1,0,0,0,1,0
1,0.77,0.53,5,256,3,0,0,0,0,1,...,0,0,0,0,0,0,0,0,1,0
2,0.89,0.79,3,149,2,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,0
3,0.64,0.63,3,156,6,1,0,0,0,0,...,0,0,0,1,0,0,0,1,0,0
4,0.98,0.74,4,151,3,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,0


In [23]:
cols_model = set(X_ohe.columns.values)
cols_score = set(X_ohe_s.columns.values)

# モデルにはあったスコアにはないデータ項目
diff1 = cols_model - cols_score
print('Modelのみ:%s' % diff1)

# スコアにはあるがモデルになかったデータ項目
diff2 = cols_score - cols_model
print('Scoreのみ:%s' % diff2)

Modelのみ:set()
Scoreのみ:set()


In [24]:
X_fin_s = X_ohe_s.loc[:,X_ohe_columns[selector.support_]]
print(X_fin_s.shape)
X_fin_s.head()

(10499, 10)


,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,sales_sales,sales_support,salary_low,salary_medium
0,0.53,0.52,2,135,4,0,0,0,0,1
1,0.77,0.53,5,256,3,0,0,0,0,1
2,0.89,0.79,3,149,2,0,0,1,0,1
3,0.64,0.63,3,156,6,1,0,1,1,0
4,0.98,0.74,4,151,3,0,1,0,0,1


In [25]:
# import libraries
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# holdout
X_train,X_test,y_train,y_test=train_test_split(X_fin_s,
                                               y,
                                               test_size=0.3,
                                               random_state=1)
# set pipelines for different algorithms
pipelines = {
    'knn':
        Pipeline([('scl',StandardScaler()),
                  ('est',KNeighborsClassifier())]),

}
# fit & evaluation
scores = {}
for pipe_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)
    scores[(pipe_name,'train')] = accuracy_score(y_train, pipeline.predict(X_train))
    scores[(pipe_name,'test')] = accuracy_score(y_test, pipeline.predict(X_test))

pd.Series(scores).unstack()

,test,train
knn,0.952063,0.964349


In [26]:
df.to_csv("20190925_sumi.csv")

In [ ]:
# パイプラインの保存
    'logistic':
        Pipeline([('scl',StandardScaler()),
                  ('est',LogisticRegression(random_state=1))]),
    'rsvc':
        Pipeline([('scl',StandardScaler()),
                  ('est',SVC(C=1.0,kernel='rbf',class_weight='balanced',random_state=1))]),
    'knn':
        Pipeline([('scl',StandardScaler()),
                  ('est',KNeighborsClassifier())]),